In [1]:
#cell 1
# Install BM25 package
!pip -q install rank_bm25

In [2]:
#cell 2
# Import libraries
import json
import re
import random
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi

In [3]:
#cell 3
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
#cell 4
# Define paths
BASE_DIR = Path("/content/drive/MyDrive")

HOTPOT_CHUNKS_PATH = BASE_DIR / "final_project/RAG/hotpotqa_docs_chunks.json"
TWOWIKI_CHUNKS_PATH = BASE_DIR / "final_project/RAG/2wikimultihopqa_docs_chunks.json"

HOTPOT_QUESTIONS_PATH = BASE_DIR / "final_project/hotpotqa_dev_2017wiki_1000_converted.json"
TWOWIKI_QUESTIONS_PATH = BASE_DIR / "final_project/2wikimultihopqa_dev_2020wiki_1000_converted.json"

HOTPOT_OUTPUT_PATH = BASE_DIR / "final_project/BM25/evidence/hotpotqa_evidence.json"
TWOWIKI_OUTPUT_PATH = BASE_DIR / "final_project/BM25/evidence/2wikimultihopqa_evidence.json"

HOTPOT_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
TWOWIKI_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

In [5]:
#cell 5
# Global settings
TOP_K = 4
TITLE_BOOST = 3
SHUFFLE_SEED = 42

In [6]:
#cell 6
# Load JSON file
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# Save JSON file
def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

In [7]:
#cell 7
# Tokenize text for BM25
def tokenize(text):
    text = str(text).lower()
    return re.findall(r"[a-z0-9]+(?:'[a-z0-9]+)?", text)

In [8]:
#cell 8
# Build BM25 corpus from chunks
def build_bm25_index(chunks, title_boost=3):
    tokenized_corpus = []
    clean_chunks = []

    for chunk in chunks:
        title = chunk.get("Title", "")
        text = chunk.get("Text", "")

        # Boost title by repetition
        indexed_text = (" ".join([title] * title_boost)) + " " + text

        tokenized_corpus.append(tokenize(indexed_text))
        clean_chunks.append({
            "title": title,
            "text": text
        })

    bm25 = BM25Okapi(tokenized_corpus)
    return bm25, clean_chunks

In [9]:
#cell 9
# Retrieve top-k chunks with BM25
def retrieve_top_k(question, bm25, clean_chunks, top_k=4):
    query_tokens = tokenize(question)

    # Score every chunk
    scores = bm25.get_scores(query_tokens)

    if len(scores) == 0:
        return []

    k = min(top_k, len(scores))

    # Fast top-k selection
    top_indices = np.argpartition(scores, -k)[-k:]
    top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]

    evidence_chunks = []
    for idx in top_indices:
        chunk = clean_chunks[int(idx)]
        evidence_chunks.append({
            "title": chunk["title"],
            "text": chunk["text"]
        })

    return evidence_chunks

In [10]:
#cell 10
# Process one dataset independently
def process_dataset(
    dataset_name,
    chunks_path,
    questions_path,
    output_path,
    top_k=4,
    title_boost=3,
    shuffle_seed=42
):
    print(f"Processing dataset: {dataset_name}")

    # Load chunks and questions
    chunks = load_json(chunks_path)
    questions_data = load_json(questions_path)

    print(f"Number of chunks: {len(chunks)}")
    print(f"Number of questions: {len(questions_data)}")

    # Build separate BM25 index
    bm25, clean_chunks = build_bm25_index(
        chunks=chunks,
        title_boost=title_boost
    )

    # Shuffle questions only
    rng = random.Random(shuffle_seed)
    shuffled_questions = questions_data.copy()
    rng.shuffle(shuffled_questions)

    results = []

    for item in tqdm(shuffled_questions, desc=f"Retrieving {dataset_name}"):
        question = item.get("question", "")

        evidence_chunk = retrieve_top_k(
            question=question,
            bm25=bm25,
            clean_chunks=clean_chunks,
            top_k=top_k
        )

        output_item = {
            "type": item.get("type"),
            "question": item.get("question"),
            "answer": item.get("answer"),
            "supports": item.get("supports"),
            "evidence_chunk": evidence_chunk
        }

        results.append(output_item)

    # Save output
    save_json(results, output_path)

    print(f"Saved: {output_path}")
    print(f"Output records: {len(results)}")

    return results

In [11]:
#cell 11
# Process HotpotQA separately
hotpot_results = process_dataset(
    dataset_name="hotpotqa",
    chunks_path=HOTPOT_CHUNKS_PATH,
    questions_path=HOTPOT_QUESTIONS_PATH,
    output_path=HOTPOT_OUTPUT_PATH,
    top_k=TOP_K,
    title_boost=TITLE_BOOST,
    shuffle_seed=SHUFFLE_SEED
)

Processing dataset: hotpotqa
Number of chunks: 35029
Number of questions: 1000


Retrieving hotpotqa:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/final_project/BM25/evidence/hotpotqa_evidence.json
Output records: 1000


In [12]:
#cell 12
# Process 2WikiMultiHopQA separately
twowiki_results = process_dataset(
    dataset_name="2wikimultihopqa",
    chunks_path=TWOWIKI_CHUNKS_PATH,
    questions_path=TWOWIKI_QUESTIONS_PATH,
    output_path=TWOWIKI_OUTPUT_PATH,
    top_k=TOP_K,
    title_boost=TITLE_BOOST,
    shuffle_seed=SHUFFLE_SEED
)

Processing dataset: 2wikimultihopqa
Number of chunks: 12685
Number of questions: 1000


Retrieving 2wikimultihopqa:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/final_project/BM25/evidence/2wikimultihopqa_evidence.json
Output records: 1000


In [13]:
#cell 13
# Validate output format
def validate_results(results, top_k=4):
    required_keys = {
        "type",
        "question",
        "answer",
        "supports",
        "evidence_chunk"
    }

    for i, item in enumerate(results):
        assert set(item.keys()) == required_keys, f"Wrong keys at index {i}"
        assert isinstance(item["evidence_chunk"], list), f"evidence_chunk is not list at index {i}"
        assert len(item["evidence_chunk"]) <= top_k, f"Too many chunks at index {i}"

        for chunk in item["evidence_chunk"]:
            assert set(chunk.keys()) == {"title", "text"}, f"Wrong evidence keys at index {i}"

    print("Validation passed.")


validate_results(hotpot_results, top_k=TOP_K)
validate_results(twowiki_results, top_k=TOP_K)

Validation passed.
Validation passed.


In [14]:
#cell 14
# Preview one HotpotQA result
hotpot_results[0]

{'type': 'comparison',
 'question': 'Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?',
 'answer': 'Bedknobs and Broomsticks',
 'supports': [['Bedknobs and Broomsticks',
   'Bedknobs and Broomsticks is a 1971 British-American musical fantasy film produced by Walt Disney Productions and released by Buena Vista Distribution Company in North America on December 13, 1971.'],
  ['The Muppet Christmas Carol',
   'The Muppet Christmas Carol is a 1992 American-British musical fantasy comedy-drama film and an adaptation of Charles Dickens\'s 1843 novel "A Christmas Carol".']],
 'evidence_chunk': [{'title': 'The Muppet Christmas Carol',
   'text': 'The Muppet Christmas Carol is a 1992 American-British musical fantasy comedy-drama film and an adaptation of Charles Dickens\'s 1843 novel "A Christmas Carol". It is the fourth in a series of live-action musical films featuring The Muppets, with Michael Caine starring as Ebenezer Scrooge. Although it is a co

In [15]:
#cell 15
# Preview one 2WikiMultiHopQA result
twowiki_results[0]

{'type': 'bridge_comparison',
 'question': 'Which film has the director died earlier, John Jaffer Janardhanan or Kamakalawa?',
 'answer': 'Kamakalawa',
 'supports': [['John Jaffer Janardhanan',
   'John Jaffer Janardhanan is a 1982 Malayalam movie directed by I. V. Sasi, written by T. Damodaran, starring Ratheesh, Ravindran and Mammootty.'],
  ['Kamakalawa',
   'Kamakalawa is a 1981 Filipino fantasy film written, produced and directed by Eddie Romero and starred Christopher De Leon, Tetchie Agbayani and Chat Silayan.'],
  ['I. V. Sasi',
   'Irruppam Veedu Sasidaran( 28 March 1948 – 24 October 2017), better known as I. V. Sasi, was an Indian film director who made over 150 films in various Indian languages.'],
  ['Eddie Romero',
   'Edgar Sinco Romero( July 7, 1924 – May 28, 2013) was a Filipino film director, film producer and screenwriter.']],
 'evidence_chunk': [{'title': 'John Jaffer Janardhanan',
   'text': 'John Jaffer Janardhanan is a 1982 Malayalam movie directed by I. V. Sasi, 

In [16]:
#cell 16
# Check saved files
print("HotpotQA output:", HOTPOT_OUTPUT_PATH)
print("2WikiMultiHopQA output:", TWOWIKI_OUTPUT_PATH)

print("HotpotQA exists:", HOTPOT_OUTPUT_PATH.exists())
print("2WikiMultiHopQA exists:", TWOWIKI_OUTPUT_PATH.exists())

HotpotQA output: /content/drive/MyDrive/final_project/BM25/evidence/hotpotqa_evidence.json
2WikiMultiHopQA output: /content/drive/MyDrive/final_project/BM25/evidence/2wikimultihopqa_evidence.json
HotpotQA exists: True
2WikiMultiHopQA exists: True


In [17]:
#cell 17
# Optional: inspect output sizes
print("HotpotQA records:", len(hotpot_results))
print("2WikiMultiHopQA records:", len(twowiki_results))

print("HotpotQA first evidence count:", len(hotpot_results[0]["evidence_chunk"]))
print("2Wiki first evidence count:", len(twowiki_results[0]["evidence_chunk"]))

HotpotQA records: 1000
2WikiMultiHopQA records: 1000
HotpotQA first evidence count: 4
2Wiki first evidence count: 4
